# 03 — RCA & Reliability Analysis

Root cause analysis, corrective actions review, and reliability KPI deep-dive.
Requires processed parquets from notebook 01.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'

from src.features import compute_availability, compute_mtbf, fleet_kpis, compute_anomaly_windows
from src.rca import failure_pareto, downtime_pareto, failure_timeline, monthly_failure_rate
from src.viz import plot_temp_trend, plot_corrective_timeline, plot_pareto

PROCESSED = repo_root / 'data' / 'processed'

In [ ]:
scada   = pd.read_parquet(PROCESSED / 'scada.parquet')
logbook = pd.read_parquet(PROCESSED / 'logbook.parquet')

## 1. Reliability KPIs per turbine

In [ ]:
avail = compute_availability(scada, logbook)
print('Availability by turbine:')
display(avail)

mtbf = compute_mtbf(logbook)
print('\nMTBF by turbine:')
display(mtbf)

In [ ]:
# Availability bar chart
if not avail.empty and 'availability_pct' in avail.columns:
    fig = px.bar(
        avail.sort_values('availability_pct'),
        x='turbine_id', y='availability_pct',
        color='availability_pct',
        color_continuous_scale='RdYlGn',
        range_color=[80, 100],
        title='Availability (%) by Turbine',
        labels={'availability_pct': 'Availability (%)', 'turbine_id': 'Turbine'},
    )
    fig.add_hline(y=95, line_dash='dash', line_color='green', annotation_text='95% target')
    fig.show()

## 2. Root cause: failure Pareto

In [ ]:
count_pareto = failure_pareto(logbook)
print('Failure count Pareto:')
display(count_pareto)

fig = plot_pareto(count_pareto, title='Failure Count Pareto — Which components fail most?')
fig.show()

In [ ]:
# Pareto by downtime impact
if 'duration_hours' in logbook.columns:
    dt_pareto = downtime_pareto(logbook)
    print('Downtime Pareto:')
    display(dt_pareto)
    fig = plot_pareto(dt_pareto, value_label='Downtime (h)',
                      title='Downtime Pareto — Which components cost the most hours?')
    fig.show()

## 3. Temperature anomaly detection (pre-failure)

In [ ]:
# Choose temperature column that's available
temp_candidates = ['gearbox_oil_temp_c', 'gearbox_bearing_temp_c', 'generator_winding_temp_c']
temp_col = next((c for c in temp_candidates if c in scada.columns), None)

if temp_col:
    # Run anomaly window computation for gearbox-related failures
    gear_failures = logbook[logbook['component'].str.contains('Gearbox', na=False)] \
                    if 'component' in logbook.columns else logbook

    scada_anomaly = compute_anomaly_windows(
        scada, gear_failures,
        temp_col=temp_col,
        window_days=7,
        ambient_col='ambient_temp_c' if 'ambient_temp_c' in scada.columns else None,
    )

    turbines = sorted(scada_anomaly['turbine_id'].unique())[:2] if 'turbine_id' in scada_anomaly.columns else [None]
    for tid in turbines:
        fig = plot_temp_trend(
            scada_anomaly,
            turbine_id=tid,
            logbook=gear_failures,
            title=f'Gearbox Temperature & Anomaly Windows — {tid}',
        )
        fig.show()
else:
    print('No temperature column found in SCADA data.')

## 4. Corrective actions timeline

In [ ]:
timeline = failure_timeline(logbook)
print(f'{len(timeline)} corrective action events')
display(timeline.head(10))

In [ ]:
from src.viz import plot_corrective_timeline
fig = plot_corrective_timeline(logbook)
fig.show()

## 5. MTBF trend analysis

In [ ]:
# MTBF over rolling 6-month windows
if 'timestamp' in logbook.columns and 'turbine_id' in logbook.columns:
    logbook_s = logbook.set_index('timestamp').sort_index()
    results = []
    for tid, grp in logbook_s.groupby('turbine_id'):
        monthly_counts = grp.resample('ME').size().reset_index(name='failures')
        monthly_counts['turbine_id'] = tid
        results.append(monthly_counts)
    if results:
        monthly_all = pd.concat(results)
        fig = px.bar(
            monthly_all,
            x='timestamp', y='failures',
            color='turbine_id',
            barmode='group',
            title='Monthly Failure Events per Turbine',
            labels={'timestamp': 'Month', 'failures': 'Failure Count'},
        )
        fig.show()

## 6. Key findings summary

Fill in after running the cells above. Suggested structure:

| Finding | Evidence | Corrective Action |
|---------|----------|-------------------|
| Top failure mode: Gearbox (XX%) | Pareto chart | Increase oil change frequency |
| Temp rise 3–5 days before gearbox failures | Z-score anomaly window | Add real-time temp alert |
| T04 availability below fleet avg | KPI table | Inspect bearing wear |
